In [1]:
import time
from datetime import datetime
from pymongo import MongoClient
from pymongo.errors import ConnectionFailure, OperationFailure

# ===== CONFIGURACIÓN DEL PROYECTO =====
NOMBRE_GRUPO = "G1_Ecommerce"
CATEGORIA_ACTUAL = "Mystery"  # Categoría asignada al grupo

# ===== CONEXIÓN A MONGODB =====
def conectar_mongodb():
    """
    Establece conexión con MongoDB y retorna la colección de trabajo.
    
    Returns:
        collection: Objeto de colección de MongoDB
    """
    try:
        client = MongoClient('database', 27017, serverSelectionTimeoutMS=5000)
        # Verificar conexión
        client.admin.command('ping')
        
        db = client['proyecto_bigdata']
        coleccion = db['datos_libros']
        
        print("✓ CONEXIÓN EXITOSA a MongoDB")
        print(f"✓ Base de datos: {db.name}")
        print(f"✓ Colección: {coleccion.name}")
        print(f"✓ Categoría configurada: {CATEGORIA_ACTUAL}\n")
        
        return coleccion
        
    except ConnectionFailure as e:
        print(f"✗ ERROR: No se pudo conectar a MongoDB - {e}")
        raise
    except Exception as e:
        print(f"✗ ERROR INESPERADO: {e}")
        raise

# Ejecutar conexión
coleccion = conectar_mongodb()

✓ CONEXIÓN EXITOSA a MongoDB
✓ Base de datos: proyecto_bigdata
✓ Colección: datos_libros
✓ Categoría configurada: Mystery



In [2]:
def procesar_y_almacenar_libros(lista_libros, coleccion):
    """
    Procesa lista de libros capturados y los almacena en MongoDB.
    
    Args:
        lista_libros (list): Lista de diccionarios con información de libros
        coleccion: Objeto de colección MongoDB
    
    Returns:
        dict: Estadísticas del proceso de inserción
    """
    estadisticas = {
        'total_procesados': 0,
        'insertados_exitosos': 0,
        'errores': 0
    }
    
    print(f"Procesando {len(lista_libros)} libros de categoría {CATEGORIA_ACTUAL}...\n")
    
    for idx, libro in enumerate(lista_libros, 1):
        try:
            # Limpiar y validar precio
            precio_limpio = libro["precio"].replace(' ', '').replace(',', '').strip()
            valor_numerico = float(precio_limpio)
            
            # Crear registro enriquecido
            registro = {
                "item": libro["titulo"],
                "valor": valor_numerico,
                "categoria": CATEGORIA_ACTUAL,
                "grupo": NOMBRE_GRUPO,
                "fecha_captura": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "timestamp": time.time()
            }
            
            # Insertar en MongoDB
            resultado = coleccion.insert_one(registro)
            estadisticas['insertados_exitosos'] += 1
            
            print(f"  [{idx}/{len(lista_libros)}] ✓ Insertado: {libro['titulo'][:50]}... (${valor_numerico})")
            
        except ValueError as e:
            print(f"  [{idx}/{len(lista_libros)}] ✗ Error de formato en precio: {libro['titulo']} - {e}")
            estadisticas['errores'] += 1
        except Exception as e:
            print(f"  [{idx}/{len(lista_libros)}] ✗ Error procesando: {libro['titulo']} - {e}")
            estadisticas['errores'] += 1
        
        estadisticas['total_procesados'] += 1
    
    # Resumen final
    print("\n" + "="*60)
    print("RESUMEN DEL PROCESO DE INSERCIÓN")
    print("="*60)
    print(f"Total procesados:      {estadisticas['total_procesados']}")
    print(f"Insertados exitosos:   {estadisticas['insertados_exitosos']}")
    print(f"Errores encontrados:   {estadisticas['errores']}")
    print(f"Tasa de éxito:         {(estadisticas['insertados_exitosos']/estadisticas['total_procesados']*100):.1f}%")
    print("="*60 + "\n")
    
    return estadisticas

# ===== DATOS DE EJEMPLO (Categoría Mystery) =====
# En producción, estos datos vendrían de Selenium/web scraping
lista_libros = [
    {"titulo": "Sharp Objects", "precio": "47.82"},
    {"titulo": "In a Dark, Dark Wood", "precio": "19.63"},
    {"titulo": "The Past Never Ends", "precio": "56.50"},
    {"titulo": "A Murder in Time", "precio": "16.64"},
    {"titulo": "The Girl You Lost", "precio": "12.29"}
]

# Ejecutar proceso de almacenamiento
stats = procesar_y_almacenar_libros(lista_libros, coleccion)

Procesando 5 libros de categoría Mystery...

  [1/5] ✓ Insertado: Sharp Objects... ($47.82)
  [2/5] ✓ Insertado: In a Dark, Dark Wood... ($19.63)
  [3/5] ✓ Insertado: The Past Never Ends... ($56.5)
  [4/5] ✓ Insertado: A Murder in Time... ($16.64)
  [5/5] ✓ Insertado: The Girl You Lost... ($12.29)

RESUMEN DEL PROCESO DE INSERCIÓN
Total procesados:      5
Insertados exitosos:   5
Errores encontrados:   0
Tasa de éxito:         100.0%



In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

print("Inicializando SparkSession...")

# ===== CONFIGURACIÓN DE SPARK =====
spark = SparkSession.builder \
    .appName("Analisis_Categorias_Mystery") \
    .config("spark.mongodb.read.connection.uri", 
            "mongodb://database:27017/proyecto_bigdata.datos_libros") \
    .config("spark.jars.packages", 
            "org.mongodb.spark:mongo-spark-connector_2.12:10.1.1") \
    .getOrCreate()

print("✓ SparkSession iniciada correctamente")
print(f"✓ Versión de Spark: {spark.version}")
print(f"✓ Configuración MongoDB: proyecto_bigdata.datos_libros\n")

Inicializando SparkSession...
✓ SparkSession iniciada correctamente
✓ Versión de Spark: 3.5.0
✓ Configuración MongoDB: proyecto_bigdata.datos_libros



In [4]:
# ===== CARGA DE DATOS DESDE MONGODB =====
print("Cargando datos desde MongoDB...\n")
df = spark.read.format("mongodb").load()

# Mostrar esquema de datos
print("ESQUEMA DE DATOS:")
print("="*60)
df.printSchema()

# Mostrar primeros registros
print("\nPRIMEROS 5 REGISTROS:")
print("="*60)
df.show(5, truncate=False)

# Estadísticas básicas
total_registros = df.count()
print(f"\nTotal de registros cargados: {total_registros}")

Cargando datos desde MongoDB...

ESQUEMA DE DATOS:
root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- timestamp: double (nullable = true)
 |-- valor: double (nullable = true)


PRIMEROS 5 REGISTROS:
+------------------------+---------+-------------------+------------+------------------------+---------+-----+
|_id                     |categoria|fecha_captura      |grupo       |item                    |timestamp|valor|
+------------------------+---------+-------------------+------------+------------------------+---------+-----+
|69e040f0c4af8f3827b816fe|Travel   |2026-04-16 01:52:48|G1_Ecommerce|It’s Only Himalaya      |NULL     |45.17|
|69e040f0c4af8f3827b816ff|Travel   |2026-04-16 01:52:48|G1_Ecommerce|Full Moon over Noahs Ark|NULL     |49.43|
|69e04159c4af8f3827b81701|Travel   |2026-04-16 01:54:33|G1_Ecommerce|It’s Only Himalaya      

In [5]:
from pyspark.sql.functions import col

print("Aplicando filtros de validación...\n")

# ===== VALIDACIÓN Y LIMPIEZA =====
# Criterios de validación:
# 1. El campo 'item' no debe ser nulo
# 2. El valor debe ser mayor a 0
# 3. Opcionalmente: filtrar solo categoría Mystery

df_filtrado = df.filter(
    (col("item").isNotNull()) & 
    (col("valor") > 0)
)

# Si queremos trabajar solo con nuestra categoría:
df_mystery = df_filtrado.filter(col("categoria") == CATEGORIA_ACTUAL)

# ===== REPORTE DE LIMPIEZA =====
print("="*60)
print("REPORTE DE VALIDACIÓN DE DATOS")
print("="*60)
print(f"Registros originales:           {df.count()}")
print(f"Registros válidos (todos):      {df_filtrado.count()}")
print(f"Registros categoría {CATEGORIA_ACTUAL}:     {df_mystery.count()}")
print(f"Registros descartados:          {df.count() - df_filtrado.count()}")
print("="*60 + "\n")

# Mostrar muestra de datos limpios
print(f"Muestra de datos de categoría {CATEGORIA_ACTUAL}:")
df_mystery.select("item", "valor", "categoria", "fecha_captura").show(5, truncate=False)

Aplicando filtros de validación...

REPORTE DE VALIDACIÓN DE DATOS
Registros originales:           32
Registros válidos (todos):      32
Registros categoría Mystery:     20
Registros descartados:          0

Muestra de datos de categoría Mystery:
+--------------------+-----+---------+-------------------+
|item                |valor|categoria|fecha_captura      |
+--------------------+-----+---------+-------------------+
|Sharp Objects       |47.82|Mystery  |2026-04-16 02:27:31|
|In a Dark, Dark Wood|19.63|Mystery  |2026-04-16 02:27:31|
|The Past Never Ends |56.5 |Mystery  |2026-04-16 02:27:31|
|A Murder in Time    |16.64|Mystery  |2026-04-16 02:27:31|
|The Girl You Lost   |12.29|Mystery  |2026-04-16 02:27:31|
+--------------------+-----+---------+-------------------+
only showing top 5 rows



In [6]:
from pyspark.sql.functions import avg, min, max, count, stddev

# ===== ANÁLISIS ESTADÍSTICO =====
print("ESTADÍSTICAS DESCRIPTIVAS - CATEGORÍA MYSTERY")
print("="*60)

stats_mystery = df_mystery.select(
    count("valor").alias("cantidad"),
    avg("valor").alias("precio_promedio"),
    min("valor").alias("precio_minimo"),
    max("valor").alias("precio_maximo"),
    stddev("valor").alias("desviacion_estandar")
).collect()[0]

print(f"Cantidad de libros:        {stats_mystery['cantidad']}")
print(f"Precio promedio:          ${stats_mystery['precio_promedio']:.2f}")
print(f"Precio mínimo:            ${stats_mystery['precio_minimo']:.2f}")
print(f"Precio máximo:            ${stats_mystery['precio_maximo']:.2f}")
print(f"Desviación estándar:      ${stats_mystery['desviacion_estandar']:.2f}")
print("="*60 + "\n")

# Top 5 libros más caros
print("TOP 5 LIBROS MÁS CAROS:")
df_mystery.orderBy(col("valor").desc()).select("item", "valor").show(5, truncate=False)

ESTADÍSTICAS DESCRIPTIVAS - CATEGORÍA MYSTERY
Cantidad de libros:        20
Precio promedio:          $30.58
Precio mínimo:            $12.29
Precio máximo:            $56.50
Desviación estándar:      $18.46

TOP 5 LIBROS MÁS CAROS:
+-------------------+-----+
|item               |valor|
+-------------------+-----+
|The Past Never Ends|56.5 |
|The Past Never Ends|56.5 |
|The Past Never Ends|56.5 |
|The Past Never Ends|56.5 |
|Sharp Objects      |47.82|
+-------------------+-----+
only showing top 5 rows

